# Example 9.5.2 / 1: Write and Read a Sheet

Companion notebook for **9.5.2 Excel Spreadsheets** in the AMPL lab Documentation, section "Write and Read a Sheet". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Write and Read a Sheet

This example writes the products to sheet `Products` of a new workbook `planning.xlsx`, clears
the data and reads the sheet back:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # maximum sales, tons

table ProductsOut OUT "amplxl" "planning.xlsx" "Products":
    PROD -> [product], profit, market;
table ProductsIn IN "amplxl" "planning.xlsx" "Products":
    PROD <- [product], profit, market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import os
for name in ("planning.xlsx", "planning.xlsx.amplbak"):
    if os.path.exists(name):
        os.remove(name)
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}

In [5]:
# run.py
ampl.eval("load amplxl.dll;")
ampl.write_table("ProductsOut")
ampl.eval("reset data;")
ampl.read_table("ProductsIn")
print(ampl.get_data("profit", "market").to_pandas())
ampl.write_table("ProductsOut")
print(sorted(f for f in os.listdir() if f.startswith("planning.xlsx")))

       profit  market
bands      25    6000
coils      30    4000
plate      29    3500
['planning.xlsx', 'planning.xlsx.amplbak']


### Pure AMPL: model.run

The sheet holds one header row (`product`, `profit`, `market`) and one row per product. The
second `write_table` finds an existing workbook, so `amplxl` first copies it to a backup file
next to it. Add the option `"backup=false"` to the declaration to skip the backup; keep backups
on while developing an output workflow.

Pure AMPL, with the same `model.mod` and data:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
import os
for name in ("planning.xlsx", "planning.xlsx.amplbak"):
    if os.path.exists(name):
        os.remove(name)
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}

In [8]:
%%ampl_eval
# model.run
load amplxl.dll;
write table ProductsOut;
reset data;
read table ProductsIn;
display profit, market;

:     profit market    :=
bands    25    6000
coils    30    4000
plate    29    3500
;

